# 04 — Model Experiments

**Goal**: Compare 4 fraud detection models with MLflow tracking

| Model | Imbalance Handling |
|-------|-------------------|
| HistGradientBoosting | `class_weight='balanced'` |
| XGBoost | `scale_pos_weight` (auto from data) |
| LightGBM | `is_unbalance=True` |
| Logistic Regression | `class_weight='balanced'` (baseline) |

**Primary metrics**: PR-AUC, F2-Score (β=2), Fraud Recall

---

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (14, 5), 'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})

from src.data.loader import load_train_test
from src.data.features import (
    engineer_features, split_features_target,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES
)
from src.models.trainers import train_model
from src.models.evaluation import (
    compute_fraud_metrics, compare_models,
    plot_pr_curve, plot_roc_curve, plot_confusion_matrix, plot_threshold_analysis
)
from src.pipeline import load_config

cfg = load_config('../configs/config.yaml')
mlflow.set_tracking_uri('../experiments/mlruns.db')
mlflow.set_experiment('credit-card-fraud-detection')

df_tr, df_te = load_train_test(
    '../data/fraudTrain.csv', '../data/fraudTest.csv', train_sample_size=200_000
)
df_tr = engineer_features(df_tr)
df_te = engineer_features(df_te)

num_feats = cfg['features']['numeric']
cat_feats = cfg['features']['categorical']
X_train, y_train = split_features_target(df_tr, feature_names=num_feats+cat_feats)
X_test,  y_test  = split_features_target(df_te, feature_names=num_feats+cat_feats)

print(f'Train: {X_train.shape} | Fraud: {y_train.mean():.4%}')
print(f'Test:  {X_test.shape}  | Fraud: {y_test.mean():.4%}')
results = {}

## E1: Logistic Regression (Baseline)

In [ ]:
res_lr = train_model(
    model_name='logistic_regression',
    X_train=X_train, y_train=y_train,
    X_val=X_test, y_val=y_test,
    model_params={'C': 0.1, 'max_iter': 500, 'class_weight': 'balanced'},
    numeric_features=num_feats, categorical_features=cat_feats,
    threshold_method='f2', beta=2.0,
    run_name='logistic_regression_baseline',
    save_path='../outputs/models/logistic_regression_model.joblib',
)
results['Logistic Regression'] = res_lr
print(f"✅ LR | PR-AUC={res_lr['val_metrics']['pr_auc']} | F2={res_lr['val_metrics']['f2_score']}")

## E2: HistGradientBoosting (sklearn, fast)

In [ ]:
res_hgb = train_model(
    model_name='hist_gradient_boosting',
    X_train=X_train, y_train=y_train,
    X_val=X_test, y_val=y_test,
    model_params={'class_weight': 'balanced', 'learning_rate': 0.05,
                  'max_iter': 200, 'max_depth': 8, 'min_samples_leaf': 20},
    numeric_features=num_feats, categorical_features=cat_feats,
    threshold_method='f2', beta=2.0,
    run_name='hist_gradient_boosting_200iter',
    save_path='../outputs/models/hist_gradient_boosting_model.joblib',
)
results['HistGradientBoosting'] = res_hgb
print(f"✅ HGB | PR-AUC={res_hgb['val_metrics']['pr_auc']} | F2={res_hgb['val_metrics']['f2_score']}")

## E3: XGBoost (auto scale_pos_weight)

In [ ]:
res_xgb = train_model(
    model_name='xgboost',
    X_train=X_train, y_train=y_train,
    X_val=X_test, y_val=y_test,
    model_params={'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
                  'subsample': 0.8, 'colsample_bytree': 0.8},
    numeric_features=num_feats, categorical_features=cat_feats,
    threshold_method='f2', beta=2.0,
    run_name='xgboost_300est',
    save_path='../outputs/models/xgboost_model.joblib',
)
results['XGBoost'] = res_xgb
print(f"✅ XGB | PR-AUC={res_xgb['val_metrics']['pr_auc']} | F2={res_xgb['val_metrics']['f2_score']}")

## E4: LightGBM (is_unbalance=True)

In [ ]:
res_lgb = train_model(
    model_name='lightgbm',
    X_train=X_train, y_train=y_train,
    X_val=X_test, y_val=y_test,
    model_params={'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
                  'subsample': 0.8, 'colsample_bytree': 0.8, 'is_unbalance': True},
    numeric_features=num_feats, categorical_features=cat_feats,
    threshold_method='f2', beta=2.0,
    run_name='lightgbm_300est_unbalance',
    save_path='../outputs/models/lightgbm_model.joblib',
)
results['LightGBM'] = res_lgb
print(f"✅ LGB | PR-AUC={res_lgb['val_metrics']['pr_auc']} | F2={res_lgb['val_metrics']['f2_score']}")

## E5: Model Comparison

In [ ]:
comparison = compare_models(results)
disp_cols = ['model','pr_auc','roc_auc','f2_score','fraud_recall','fraud_precision','threshold','cv_pr_auc_mean']
print('=== MODEL COMPARISON (sorted by PR-AUC) ===')
print(comparison[[c for c in disp_cols if c in comparison.columns]].to_string(index=False))

In [ ]:
# Bar chart comparison
COLORS5 = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
model_names = list(results.keys())

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, metric, col in zip(axes,
    ['pr_auc', 'f2_score', 'fraud_recall'],
    ['PR-AUC', 'F2-Score', 'Fraud Recall']):
    vals = [r['val_metrics'][metric] for r in results.values()]
    bars = ax.bar(model_names, vals, color=COLORS5, alpha=0.85)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_ylim(min(vals)*0.9, 1.0)
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# PR curves
models_probs = {}
for name, res in results.items():
    y_prob = res['pipeline'].predict_proba(X_test)[:, 1]
    models_probs[name] = y_prob

fig = plot_pr_curve(y_test.values, models_probs)
plt.show()

fig = plot_roc_curve(y_test.values, models_probs)
plt.show()

In [ ]:
# Best model: confusion matrix + threshold analysis
best_name = max(results, key=lambda k: results[k]['val_metrics']['pr_auc'])
best_res   = results[best_name]
best_pipe  = best_res['pipeline']
best_thr   = best_res['threshold']
y_prob_best = best_pipe.predict_proba(X_test)[:, 1]
y_pred_best = (y_prob_best >= best_thr).astype(int)

print(f'\n🏆 Best model: {best_name}')
for k, v in best_res['val_metrics'].items():
    if isinstance(v, float): print(f'   {k:<25} {v:.4f}')

fig = plot_confusion_matrix(y_test.values, y_pred_best, model_name=best_name)
plt.show()

fig = plot_threshold_analysis(y_test.values, y_prob_best)
plt.show()

## Summary

> **MLflow**: `mlflow ui --backend-store-uri sqlite:///experiments/mlruns.db`  
> **Next**: → `05_Model_Interpretation.ipynb` — SHAP analysis